# Kaza — Phase 1 Silence & Sampling Diagnostics

Tests `best.pt` on named held-out audio organized by language on Google Drive to separate data silence from sampling collapse.

### Cell 1 — Environment setup
Mounts Google Drive, loads `HF_TOKEN` from Colab Secrets, pulls the latest `phase1-design` branch, and installs dependencies.

In [ ]:
import os, subprocess
from google.colab import drive, userdata
drive.mount("/content/drive")
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

REPO = "/content/kaza"
if os.path.isdir(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only", "origin", "phase1-design"], check=True)
else:
    subprocess.run(["git", "clone", "-b", "phase1-design", "https://github.com/udohchuks/kaza.git", REPO], check=True)
%cd /content/kaza
!pip install -q -r requirements.txt

### Cell 2 — Load codec and `best.pt`
Loads the frozen `LiquidCodec` and restores `KazaModel` weights from `best.pt` (reading `model_settings` directly from the checkpoint).

In [ ]:
from pathlib import Path
import torch
from kaza.env import print_env
from kaza.codec import FRAME_RATE, SAMPLE_RATE, SAMPLES_PER_FRAME, KazaCodec, to_mono_24k
from kaza.model import KazaModel

DRIVE_ROOT = Path("/content/drive/MyDrive/kaza_phase1")
RUN_NAME = "waxal_nope_v1"
VAL_AUDIO_DIR = DRIVE_ROOT / "val_audio"

env = print_env()
codec = KazaCodec(device=env.device, dtype=env.frozen_dtype)

ckpt = torch.load(DRIVE_ROOT / "runs" / RUN_NAME / "best.pt", map_location="cpu", weights_only=False)
ms = ckpt.get("model_settings", {})
model = KazaModel(
    device=env.device, frozen_dtype=env.frozen_dtype, trainable_dtype=env.trainable_dtype,
    autocast_dtype=env.autocast_dtype, drop_prob=ms.get("drop_prob", 0.08),
    mtp_offsets=tuple(ms.get("mtp_offsets", (2, 4))), mtp_weight=ms.get("mtp_weight", 0.2),
    nope_new_layers=ms.get("nope_new_layers", True),
)
model.load_trainable_state_dict(ckpt["model"])
model.eval()
print(f"Loaded best.pt (step={ckpt['step']}, val_k12_loss={ckpt['best_val_loss']:.4f})")

### Cell 3 — Save held-out validation clips to Drive (by language)
Streams up to `CLIPS_PER_SOURCE` natural utterances (≥ 2.5 s) per WAXAL validation split and saves them under `val_audio/<lang>/<config>_<id>.wav`. Skips sources already saved and prints every relative path so you can paste any path into Cell 5 as `AUDIO_ID`.

In [ ]:
import soundfile as sf
from datasets import Audio as HFAudio, load_dataset
from kaza.data import _decode_row_audio, _language_of, discover_waxal_sources

LANGUAGES = None       # None = all WAXAL val languages, or e.g. ["hau", "ibo", "yor", "swa"]
CLIPS_PER_SOURCE = 3

train_sources, train_weights, val_sources = discover_waxal_sources("google/WaxalNLP", include_tts=True, lang_alpha=0.5)
for dataset_id, config, split in val_sources:
    lang = _language_of(config)
    if LANGUAGES and lang not in LANGUAGES:
        continue
    lang_dir = VAL_AUDIO_DIR / lang
    lang_dir.mkdir(parents=True, exist_ok=True)
    if len(list(lang_dir.glob(f"{config}_*.wav"))) >= CLIPS_PER_SOURCE:
        continue
    ds = load_dataset(dataset_id, config, split=split, streaming=True).cast_column("audio", HFAudio(decode=False))
    saved = 0
    for idx, row in enumerate(ds):
        wav = _decode_row_audio(row)
        if wav is None or len(wav) < int(2.5 * SAMPLE_RATE):
            continue
        clip_id = str(row.get("id") or Path(row["audio"].get("path") or f"{idx}").stem).replace("/", "_")
        sf.write(lang_dir / f"{config}_{clip_id}.wav", wav[: int(12.0 * SAMPLE_RATE)].numpy(), SAMPLE_RATE)
        saved += 1
        if saved >= CLIPS_PER_SOURCE:
            break

for p in sorted(VAL_AUDIO_DIR.rglob("*.wav")):
    print(p.relative_to(VAL_AUDIO_DIR))

### Cell 4 — Silence measurement (training stream vs. saved validation clips)
Measures the percentage of 80 ms frames below `-50 dBFS` (near-digital silence / stitched gaps), `-40 dBFS` (silence/pauses), and `-35 dBFS` (quiet room tone).

In [ ]:
from kaza.data import WaxalStreamingDataset

def frame_db(wav: torch.Tensor) -> torch.Tensor:
    """Return RMS loudness in dBFS for each 80 ms codec frame (1,920 samples at 24 kHz)."""
    n = wav.numel() // SAMPLES_PER_FRAME
    rms = wav[: n * SAMPLES_PER_FRAME].reshape(n, SAMPLES_PER_FRAME).pow(2).mean(-1).sqrt()
    return 20.0 * torch.log10(rms.clamp_min(1e-8))

def report_silence(wav: torch.Tensor, label: str) -> None:
    """Print duration and fraction of 80 ms frames below -50, -40, and -35 dBFS."""
    db = frame_db(wav)
    print(f"{label:<22} | {len(db)/FRAME_RATE:6.1f}s | <-50dB: {(db<-50).float().mean()*100:5.1f}% | <-40dB: {(db<-40).float().mean()*100:5.1f}% | <-35dB: {(db<-35).float().mean()*100:5.1f}%")

it = iter(WaxalStreamingDataset(train_sources, source_weights=train_weights, clip_frames=500))
report_silence(torch.cat([next(it) for _ in range(32)]), "Train stream (32 clips)")
val_wavs = [to_mono_24k(*sf.read(p)) for p in sorted(VAL_AUDIO_DIR.rglob("*.wav"))]
report_silence(torch.cat(val_wavs), "Saved val_audio (all)")

### Cell 5 — Pick an audio clip by path (`AUDIO_ID`) and sweep sampling settings
Set `AUDIO_ID` to any relative path printed in Cell 3 (e.g. `"swa/swa_asr_0.wav"`). Encodes the clip once, feeds the first `PROMPT_SEC` to `best.pt`, and generates `GEN_SEC` across greedy (`temperature=0.0`), unfiltered, `top_p`, `top_k`, higher temperature, and more MaskGIT passes (`num_passes=6, 12`). Reports active speech % in early (`2–4 s`) vs. late (`4–8 s`) generation and late token repetition %.

In [ ]:
import torch.nn.functional as F
from IPython.display import Audio, display
from kaza.depth_decoder import MASK_TOKEN_ID
from kaza.embeddings import NUM_CODEBOOKS, shift_agent_tokens

AUDIO_ID = None   # <--- paste a path from Cell 3, e.g. "swa/swa_asr_0.wav" (None = first file)
PROMPT_SEC, GEN_SEC = 2.0, 6.0

clip_path = (VAL_AUDIO_DIR / AUDIO_ID) if AUDIO_ID else sorted(VAL_AUDIO_DIR.rglob("*.wav"))[0]
clip_wav = to_mono_24k(*sf.read(clip_path))
P, G = int(round(PROMPT_SEC * FRAME_RATE)), int(round(GEN_SEC * FRAME_RATE))
prompt = codec.encode(clip_wav)[0][:P].unsqueeze(0).to(model.device).long()

print(f"Testing: {clip_path.relative_to(VAL_AUDIO_DIR)} ({len(clip_wav)/SAMPLE_RATE:.2f}s)")
display(Audio(clip_wav.numpy(), rate=SAMPLE_RATE))

def filter_logits(logits: torch.Tensor, top_k: int | None = None, top_p: float | None = None) -> torch.Tensor:
    """Mask out low-probability tail logits using top-k and/or nucleus (top-p) truncation."""
    if top_k:
        logits = logits.masked_fill(logits < logits.topk(top_k, dim=-1).values[..., -1:], float("-inf"))
    if top_p:
        s_logits, s_idx = logits.sort(dim=-1, descending=True)
        remove = torch.softmax(s_logits, dim=-1).cumsum(dim=-1) > top_p
        remove[..., 1:], remove[..., 0] = remove[..., :-1].clone(), False
        logits = torch.full_like(logits, float("-inf")).scatter(-1, s_idx, s_logits.masked_fill(remove, float("-inf")))
    return logits

@torch.no_grad()
def generate_filtered(
    prompt_tokens: torch.Tensor,
    num_new_frames: int,
    num_passes: int = 3,
    temperature: float = 1.0,
    top_p: float | None = None,
    top_k: int | None = None,
) -> torch.Tensor:
    """Autoregressive generation with MaskGIT decoding plus optional top_p/top_k filtering.

    Uses greedy argmax when temperature <= 0.0 (deterministic), otherwise samples from the
    temperature-scaled, top_k/top_p-filtered softmax with decaying Gumbel confidence noise.
    """
    dd = model.depth_decoder
    with model._autocast():
        pkv = model.backbone(inputs_embeds=model.embeddings(shift_agent_tokens(prompt_tokens)), use_cache=True).past_key_values
        last_tok, out_frames = prompt_tokens[:, -1:, :], []
        for _ in range(num_new_frames):
            out = model.backbone(inputs_embeds=model.embeddings(last_tok), past_key_values=pkv, use_cache=True)
            pkv = out.past_key_values
            ctx = out.last_hidden_state.reshape(-1, out.last_hidden_state.shape[-1])
            tok = torch.full((ctx.shape[0], dd.num_codebooks), MASK_TOKEN_ID, dtype=torch.long, device=ctx.device)
            unmasked, rem = torch.zeros_like(tok, dtype=torch.bool), dd.num_codebooks
            for step in range(num_passes):
                k = rem if step == num_passes - 1 else rem // (num_passes - step)
                logits = dd.forward(ctx, tok).float()
                if temperature <= 0.0:
                    probs = F.softmax(logits, dim=-1)
                    pred = logits.argmax(dim=-1)
                    conf = probs.gather(-1, pred.unsqueeze(-1)).squeeze(-1)
                else:
                    probs = F.softmax(filter_logits(logits / temperature, top_k, top_p), dim=-1)
                    pred = torch.multinomial(probs.reshape(-1, dd.vocab_size), 1).reshape(ctx.shape[0], dd.num_codebooks)
                    log_p = probs.gather(-1, pred.unsqueeze(-1)).squeeze(-1).clamp_min(1e-12).log()
                    u = torch.rand_like(log_p).clamp(1e-6, 1.0 - 1e-6)
                    conf = log_p + (1.0 - step / num_passes) * (-(-u.log()).log())
                commit = torch.zeros_like(unmasked).scatter_(1, conf.masked_fill(unmasked, float("-inf")).topk(k, dim=-1).indices, True)
                tok, unmasked, rem = torch.where(commit, pred, tok), unmasked | commit, rem - k
            last_tok = tok.reshape(1, 1, NUM_CODEBOOKS)
            out_frames.append(last_tok)
    return torch.cat([prompt_tokens, *out_frames], dim=1)

grid = [
    # Deterministic & unfiltered baselines
    dict(temperature=0.0, top_p=None, top_k=None, num_passes=3),
    dict(temperature=0.8, top_p=None, top_k=None, num_passes=3),
    dict(temperature=1.0, top_p=None, top_k=None, num_passes=3),
    # Nucleus (top_p) across temperatures
    dict(temperature=0.8, top_p=0.90, top_k=None, num_passes=3),
    dict(temperature=0.9, top_p=0.85, top_k=None, num_passes=3),
    dict(temperature=1.0, top_p=0.80, top_k=None, num_passes=3),
    dict(temperature=1.0, top_p=0.90, top_k=None, num_passes=3),
    dict(temperature=1.0, top_p=0.95, top_k=None, num_passes=3),
    dict(temperature=1.2, top_p=0.90, top_k=None, num_passes=3),
    # Top-k & combined top_p + top_k
    dict(temperature=1.0, top_p=None, top_k=50,   num_passes=3),
    dict(temperature=1.0, top_p=None, top_k=200,  num_passes=3),
    dict(temperature=1.0, top_p=0.90, top_k=100,  num_passes=3),
    # More MaskGIT refinement passes
    dict(temperature=1.0, top_p=0.90, top_k=None, num_passes=6),
    dict(temperature=1.0, top_p=0.90, top_k=None, num_passes=12),
]

mid = P + G // 3
for cfg_s in grid:
    torch.manual_seed(0)
    gen = generate_filtered(prompt, G, **cfg_s)
    wav = codec.decode(gen)[0]
    db = frame_db(wav)
    rep = (gen[0, mid + 1 :] == gen[0, mid:-1]).float().mean() * 100
    print(f"{cfg_s} | active 2-4s={(db[P:mid]>=-40).float().mean()*100:5.1f}% | 4-8s={(db[mid:]>=-40).float().mean()*100:5.1f}% | rep={rep:4.1f}%")
    display(Audio(wav.numpy(), rate=SAMPLE_RATE))